## 5. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Default rate by categorical features
features_to_plot = ['purpose', 'emp_length', 'home_ownership', 'addr_state']
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    if col in df_cleaned.columns:
        risk_rates = df_cleaned.groupby(col)['default'].mean().sort_values(ascending=False).reset_index()
        risk_rates['default'] = risk_rates['default'] * 100
        sns.barplot(data=risk_rates, x=col, y='default', ax=axes[i], palette='mako', hue=col, legend=False)
        axes[i].set_title(f'Default Rate by {col.replace("_", " ").title()}', fontsize=14)
        axes[i].set_ylabel('Default Rate (%)')
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
df_cleaned.groupby('grade')['default'].mean().sort_values().plot(kind='bar',
    title='Default Rate by Grade', ylabel='Default Rate')
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px
state_dr = df_cleaned.groupby('addr_state')['default'].mean().reset_index()
state_dr['default_rate_pct'] = state_dr['default'] * 100
fig = px.choropleth(state_dr, locations='addr_state', locationmode='USA-states',
                    color='default_rate_pct', scope='usa',
                    labels={'default_rate_pct': 'Default Rate (%)'},
                    title='Default Rate by State')
fig.show()

### Numerical Feature Distribution Analysis

Visualize how key continuous features separate defaulters (red) from non-defaulters (blue).

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()
palette_colors = {0: "#1f77b4", 1: "#d62728"}

if 'fico_range_low' in df_cleaned.columns:
    sns.kdeplot(data=df_cleaned, x='fico_range_low', hue='default',
                common_norm=False, fill=True, alpha=0.4, palette=palette_colors, ax=axes[0])
    axes[0].set_title('FICO Score Distribution', fontsize=13, fontweight='bold')

if 'int_rate' in df_cleaned.columns:
    sns.kdeplot(data=df_cleaned, x='int_rate', hue='default',
                common_norm=False, fill=True, alpha=0.4, palette=palette_colors, ax=axes[1])
    axes[1].set_title('Interest Rate Distribution', fontsize=13, fontweight='bold')

if 'dti' in df_cleaned.columns:
    sns.kdeplot(data=df_cleaned[df_cleaned['dti'] <= 50], x='dti', hue='default',
                common_norm=False, fill=True, alpha=0.4, palette=palette_colors, ax=axes[2])
    axes[2].set_title('DTI Ratio (Capped at 50)', fontsize=13, fontweight='bold')

if 'annual_inc' in df_cleaned.columns:
    sns.kdeplot(data=df_cleaned[df_cleaned['annual_inc'] > 0], x='annual_inc', hue='default',
                common_norm=False, fill=True, alpha=0.4, palette=palette_colors, log_scale=True, ax=axes[3])
    axes[3].set_title('Annual Income (Log Scale)', fontsize=13, fontweight='bold')

if 'loan_amnt' in df_cleaned.columns:
    sns.histplot(data=df_cleaned, x='loan_amnt', hue='default',
                 multiple="layer", bins=35, palette=palette_colors, ax=axes[4], element="step", fill=True, alpha=0.3)
    axes[4].set_title('Loan Amount Distribution', fontsize=13, fontweight='bold')

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

**Key observations:**
- **Interest Rate** has the best separation. Defaulters skew right (15–25%), paid loans skew left (5–12%).
- **FICO Score:** Default peaks ~650–670, paid peaks ~700+. Strong signal.
- **DTI:** Mild separation — defaulters shift slightly right.
- **Annual Income:** Partial separation; log scale needed due to skew.

In [ ]:
core_numeric_cols = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'default'
]
available_cols = [col for col in core_numeric_cols if col in df_cleaned.columns]
correlation_matrix = df_cleaned[available_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Correlation Heatmap: Continuous Risk Features', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Key insights:** `fico_range_low` and `fico_range_high` are perfectly collinear (r=1.0) — drop one. `loan_amnt` and `installment` are highly correlated (~0.95) — drop `installment`.